<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 1 — Euler vs RK4: Which One Do You Trust?</div>
</div>

**Nothing here is assessed.** Have a go, compare notes with those around you, and be ready to talk a facilitator through what you found (this is good practice for the in-class assessment sessions later in the course).

## Where this fits

Today's lecture showed you the update rules for Euler's method and the classic 4th-order
Runge–Kutta method (RK4). You now know *how* each one steps a state forward in time. _Which one should you trust, and when?_

Today we'll implement both and run them on a problem whose exact answer we know. 

## <span style="color:#B5121B">0. The problem: radioactive decay</span>

Radioactive decay has a known, exact solution:

$$\frac{dN}{dt} = -\lambda N, \qquad N(t) = N_0 e^{-\lambda t}$$

This cell sets up $N_0$, $\lambda$, and a Python function that describes the *exact* solution. This will give us a ground truth to compare against.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N0 = 500.0      # initial quantity, arbitrary units
lam = 0.8       # decay constant, per second
half_life = np.log(2) / lam

print(f"lambda = {lam} /s, half-life = {half_life:.3f} s")


def N_exact(t, N0=N0, lam=lam):
    # The analytic solution -- our ground truth for everything below.
    return N0 * np.exp(-lam * t)


t_plot = np.linspace(0, 6, 400)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_plot, N_exact(t_plot), lw=2, label="exact")
ax.set_xlabel("t (s)")
ax.set_ylabel("N(t)")
ax.legend()
fig.tight_layout()
plt.show()

## <span style="color:#B5121B">1. Implement Euler's method</span>

Recall the update rule from the lecture:

$$N_{i+1} = N_i + h \cdot f(t_i, N_i), \qquad f(t, N) = -\lambda N$$

Write a function that takes a step size `h` and a total time `t_max`, and returns arrays of `t` and `N` from $t=0$ to `t_max`, starting from `N0`.

In [ ]:
def f(t, N, lam=lam):
    # dN/dt, as a function of (t, N) -- the lecture's f(t, y)
    return -lam * N


def euler_integrate(h, t_max, N0=N0, lam=lam):
    n_steps = int(round(t_max / h))
    t = np.zeros(n_steps + 1)
    N = np.zeros(n_steps + 1)
    N[0] = N0

    # TODO: fill in the Euler update loop.
    # for i in range(n_steps):
    #     t[i+1] = ...
    #     N[i+1] = ...
    raise NotImplementedError("implement the Euler loop above")

    return t, N


# Quick sanity check once you've filled it in:
# t_e, N_e = euler_integrate(h=0.2, t_max=6)
# plt.plot(t_e, N_e, "o-", ms=3, label="Euler, h=0.2")
# plt.plot(t_plot, N_exact(t_plot), label="exact")
# plt.legend(); plt.xlabel("t (s)"); plt.ylabel("N(t)"); plt.show()

## <span style="color:#B5121B">2. Implement RK4</span>

Same idea, but using the four-slope RK4 update from the lecture:

$$k_1 = f(t_i, N_i), \quad k_2 = f(t_i + h/2,\, N_i + \tfrac{h}{2}k_1), \quad
k_3 = f(t_i + h/2,\, N_i + \tfrac{h}{2}k_2), \quad k_4 = f(t_i + h,\, N_i + h k_3)$$
$$N_{i+1} = N_i + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

In [ ]:
def rk4_integrate(h, t_max, N0=N0, lam=lam):
    n_steps = int(round(t_max / h))
    t = np.zeros(n_steps + 1)
    N = np.zeros(n_steps + 1)
    N[0] = N0

    # TODO: fill in the RK4 update loop, using f(t, N) defined above.
    # for i in range(n_steps):
    #     k1 = ...
    #     k2 = ...
    #     k3 = ...
    #     k4 = ...
    #     t[i+1] = ...
    #     N[i+1] = ...
    raise NotImplementedError("implement the RK4 loop above")

    return t, N

## <span style="color:#B5121B">3. Round 1: same step size, head to head</span>

Run both methods with the **same, moderate** step size (`h = 0.2` s, `t_max = 6` s, about seven half-lives) and plot both against the exact solution on the same axes.

Can you tell the two methods apart by eye? If not, note that too: it tells you how large a difference has to be before it is visible.

In [ ]:
# TODO: call euler_integrate and rk4_integrate with h=0.2, t_max=6,
# and plot both curves together with the exact solution.


## <span style="color:#B5121B">4. Round 2: which one is closer?</span>

A plot only tells you so much. Compute the **error at the final time**, `abs(N_method[-1] - N_exact(t_max))`, for both methods at `h = 0.2`. Which is closer?

Repeat with a **coarser** step, `h = 1.0`. Does the gap between the methods grow or shrink as `h` increases? Can you explain this from the update rules?

In [ ]:
# TODO: compute and print the final-time error for Euler and RK4 at h=0.2 and h=1.0.


## <span style="color:#B5121B">5. Round 3: how does the error scale with step size?</span>

Sweep `h` over several values (`h = 1.0, 0.5, 0.25, 0.125, 0.0625`), record the final-time error for each method, and plot **error against `h` on log-log axes**.

A straight line on a log-log plot means the error scales as a power of `h`, and the slope gives the power. This is the *convergence* from the lecture: compare the slopes for the two methods.

In [ ]:
# TODO: sweep h over [1.0, 0.5, 0.25, 0.125, 0.0625], record the final-time error for
# both Euler and RK4 at each h, and plot error vs h on a log-log scale (plt.loglog).
h_values = [1.0, 0.5, 0.25, 0.125, 0.0625]


## <span style="color:#B5121B">6. Round 4: break it</span>

Convergence is only half of the story. The other half is **stability**, also from the lecture. Forward Euler applied to this equation is only stable if $h < 2/\lambda$. With $\lambda = 0.8$/s, that is $h < 2.5$ s.

Try `h = 2.6` and `h = 4.0` for **both** methods, integrating to `t_max = 20` s, and plot the results.

What happens to Euler? This isn't a small numerical error: a quantity that should only decay starts to oscillate and grow. Does RK4 fail in the same way at these step sizes?

In [ ]:
# TODO: integrate with h=2.6 and h=4.0 out to t_max=20 for both methods, and plot.
# Look closely at Euler's behaviour -- does it still look like decay?


## The verdict: in your own words

Write two or three sentences in your logbook answering:

1. At a sensible step size, which method is more accurate, and by how much? (Use the power of `h` from Section 5, not just "RK4 is better".)
2. What goes wrong with Euler once `h` crosses the stability threshold? Is it a gradually growing error, or something more sudden?
3. Why might you still choose Euler over RK4? (Hint: how much does each step cost to compute?)

Be ready to explain your answers to a facilitator.

## Looking ahead to Exercise 1

Exercise 1 (25%, released after next week's lab) uses the same decay model, $N(t) = N_0 e^{-\lambda t}$, with noisy count data and two different techniques (root-finding and curve-fitting). As today, you will compare two methods: do they agree, and if not, why not?

## <span style="color:#FD9029">Extension (optional): a two-body orbit, and a look ahead to Kepler's equation</span>

You met this system in your Year 2 course: a body orbiting under an inverse-square force. The physics isn't new; the new part is comparing how Euler and RK4 handle it.

Unlike the decay problem, the position and velocity should trace out the same closed orbit forever. This gives a visual test: does the method stay on the orbit it started on?

### The physical setup (provided)

Units are chosen so that $GM = 1$. A body starting at $(x, y) = (1, 0)$ with velocity $(v_x, v_y) = (0, 1)$ moves on a circle of radius 1, with period $T = 2\pi$. The equations of motion are:

$$\dot{x} = v_x, \qquad \dot{y} = v_y, \qquad \dot{v}_x = -\frac{GM\,x}{r^3}, \qquad \dot{v}_y = -\frac{GM\,y}{r^3}, \qquad r = \sqrt{x^2 + y^2}$$

In [ ]:
GM = 1.0
state0 = (1.0, 0.0, 0.0, 1.0)  # x, y, vx, vy -- a circular orbit, r=1, period 2*pi
T = 2 * np.pi


def orbit_accel(x, y, GM=GM):
    r3 = (x**2 + y**2) ** 1.5
    return -GM * x / r3, -GM * y / r3


def orbit_deriv(state, GM=GM):
    # state = (x, y, vx, vy) -- same idea as f(t, N) above, just a vector now.
    x, y, vx, vy = state
    ax, ay = orbit_accel(x, y, GM=GM)
    return np.array([vx, vy, ax, ay])

### Implement Euler and RK4 for the vector state

Same update rules as Sections 1–2, just applied to the 4-component state vector `(x, y, vx, vy)` instead of the single value `N`.

In [ ]:
def euler_orbit(h, n_periods, state0=state0):
    n_steps = int(round(n_periods * T / h))
    states = np.zeros((n_steps + 1, 4))
    states[0] = state0

    # TODO: fill in the Euler update loop (compare to euler_integrate above).
    # for i in range(n_steps):
    #     states[i+1] = states[i] + h * orbit_deriv(states[i])
    raise NotImplementedError("implement the Euler loop above")

    return states


def rk4_orbit(h, n_periods, state0=state0):
    n_steps = int(round(n_periods * T / h))
    states = np.zeros((n_steps + 1, 4))
    states[0] = state0

    # TODO: fill in the RK4 update loop (compare to rk4_integrate above).
    # for i in range(n_steps):
    #     k1 = orbit_deriv(states[i])
    #     k2 = orbit_deriv(states[i] + h/2*k1)
    #     k3 = orbit_deriv(states[i] + h/2*k2)
    #     k4 = orbit_deriv(states[i] + h*k3)
    #     states[i+1] = states[i] + h/6*(k1 + 2*k2 + 2*k3 + k4)
    raise NotImplementedError("implement the RK4 loop above")

    return states

### Run it: does the orbit stay put?

Integrate both methods for 5 orbital periods at `h = 0.05` (about 126 steps per orbit, deliberately coarse). Plot:

1. the orbit in the $x$–$y$ plane for both methods on the same axes (use `ax.set_aspect("equal")` so that a circle looks circular);
2. the distance from the centre, $r(t) = \sqrt{x^2+y^2}$, against time for both methods. It should stay at $r=1$.

What does Euler's orbit do?

In [ ]:
# TODO: call euler_orbit and rk4_orbit with h=0.05, n_periods=5, and make the two
# plots described above (the x-y orbit shape, and r(t) over time, for both methods).


### The verdict

1. What does Euler's orbit do over 5 periods? Does it just become *less accurate*, as in the decay problem, or does it fail in a different way?
2. Try a smaller step size (`h = 0.01`). Does Euler's problem go away, or just take longer to appear?
3. Why might an error in the *shape* of a solution (an orbit spiralling outward, energy not conserved) matter more in some applications than an error in its *size* (a slightly wrong decay curve)?

### Looking ahead to Week 2

Here the orbit is found by stepping through time. For this problem there is also a direct relation between time and position, **Kepler's equation**, $M = E - e\sin E$, where $M$ is proportional to time and the eccentric anomaly $E$ gives the position. It can't be rearranged for $E$, so it has to be solved numerically: this is Week 2's root-finding example.

*(Optional stretch: the orbit above is circular, $e=0$. Try `state0 = (1.0, 0.0, 0.0, 0.7)` for an eccentric orbit. Does Euler do better or worse?)*

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>